# __Agents__
An Agent is an entity that is expected to operate autonomously, utilizing an LLM as its reasoning "brain". The main characteristics of agents include goal orientation, planning and reasoning, perception, and adaptability. It maps a high dimensional space (the context window) to an action space (tool executions or text generation). One bottleneck in agentic architectures is the compounding error rate over sequential decision-making rollouts.

The ideia is that we do not need to dictated exactly how to do something, we just provide the goal, and the agent autonomously perceive the environment, select the tools, and adapt the plan if needed.

The data flow of an agentic loop is typically: `User Query -> State Initialization -> LLM Inference (reasoning) -> Tool Calling -> Environment Feedback -> Context Augmentation`

Some important things:
- __Goal Orientation and Planning:__ Agents work towards specfic objectives rather than simply responding to input prompts, breaking down complex problems through iteractive reasoning.
- __Perception and Action:__ Agents gather external data and manipulate their environment by invoking APIs, which is called _Tool Calling_.
- __Adaptability:__ Agents evaluate their own outputs and self-correct using reflection and learning mechanisms when encountering errors or hallucinations.

### __Multi-Agent Orchestration__
For high complex systems, single agents (monolithic) fail due to context size and conflicting prompt instructions. To handle this problem, we implement _Multiagent Collaboration_, deploying systems of various specialized agents. They converse either by updating a shared state graph or via direct message passing using JSON schemas. Typically, a "_Router_" agent classifies the intent and delegates the context payload to a specialized "_Worker_" agent, which executes the task.

### __About Implementation__
We should, ever possible, avoid using wrappers. The best way for production grade applications is to use Python Workflows, utilizing `asyncio` for I/O-bound tool executions. Our oschestration layer should be a framework-agnostic that directly wraps raw HTTP calls to our LLM provider. This ensure control over the context window, prompt construction, and system observability.

# __Tools and Tool Calling__
An LLM operates through an autoregressive policy, mapping a highly dimensional context state to a probability distribution over a discrete vocabulary. Because its parametric memory is static and its output is stochastic, it fails at deterministic computation and retrieving real-time data.

With _Tools_, whe expand the agent's action space and capability with executable tools.

In short, Tools bridge LLMs with software APIs, allowing them to invoke external functions to bypass inherent limitations like static knowledge cutoffs. "_Tool Calling_" is the specific mechanism where an LLM is prompted with functions signatures (like JSON schemas) and outputs a structured representation of the function to invoke alongside its parameters.

A _Tool_ (or Tool Function) is a backend script (like an HTTP request or a SQL query) exposed to the LLM via JSON schema. "_Tool Calling_", as mentioned above, is the mechanism where a model recognizes that a user's intent requires external computation. It halts standard generation and instead outputs a structured payload containing the exact arguments required by our function.

One architectural example: `User Query -> Prompt and Schema Injection -> LLM inference -> Intent Match -> Parameter Extraction -> Async Tool Execution -> Context Augmentation -> Resumed Inference -> Output Guardrails`
- __Schema Definition:__ We should map raw Python functions to JSON schemas manually or via lightweight parsing.
- __Async and Concurrency:__ LLM inference is computed-bound, but tool execution is heavily I/O-bound. We should execute tools using `asyncio` to prevent thread blocking, addressing TTFT (time-to-first-token) latency bottlenecks.
- __Guardrails:__ We should never trust the LLM's parameter payload. A good practice is to validate all extracted arguments using strict typing before triggering the backend function to prevent type errors and injection attacks.
- __Idempotency:__ We need to expect network faults. With that in mind, tool functions that mutate state (writes) must be idempotent so that an agent's internal retry loop does not corrupt the database.

# __Model Context Protocol (MCP)__
MCP is an open standard that stablishes a universal, secure communication layer between AI models and external data sources or tools.

_Interesting:_ Before MCP, AI systems suffered from an `O(M * N)` integration bottleneck: M different agents had to write custom API wrappers for N different data sources. With MCP, the complexity was reduced to `O(M + N)`.

MCP works as a "USB-C port" for agents, it provides plug-and-play abstraction for AI context.

### __Architecture and Execution__
MCP operates on a CLient-Server architecture, typically communicating via JSON-RPC over local standard input/output (`stdio`) or Server-Sent Events (`SSE`) for remote networks.
- __MCP Host:__ The application initiating the connection, such as an IDE or custom agentic orchestration loop.
- __MCP Clients:__ The protocol layer inside the Host that routes the requests and manages connections.
- __MCP Servers:__ Lightweight, isolated backend scripts directly connected to specific data silos (like PostgreSQL database, GitHub repo, etc).

### __Resources, Prompts, and Tools__
MCP standardizes three fundamental blocks:
- __Resources:__ Exposes static or dynamic data as URI-addressable payloads. This represents the agent's read-only perception of its environment.
- __Prompts:__ Reusable, parameterized prompt templates managed by the server to standardize context augmentation.
- __Tools:__ Deterministic, executable backend functions exposed via JSON schemas. This standardizes the agent's action space, allowing it to mutate safely.

For autonomous agents, MCP is the abstraction layer. It cleanly decouples the agent's stochastic policy (the LLM reasoning loop) from the deterministic data retrieval and execution layer. As the system scales, this separation layer of concerns ensures that an agent can dynamically discover and interact entirely new databases at runtime without requiring us to rewrite the core orchestration code.

# __Tools vs. MCP__
__Tools__ represents the action space available to the LLM. When an autoregressive policy determines that its internal parametric memory is insufficient to compute the next state, it emits a structured payload (JSON) to trigger a backend function. This means that tools are actual actions that the LLM can call, like `search_database` or `restart_server`.

__MCP__ is a universal, open-standard transport layer (based on JSON-RPC) that dictates _how_ agents connect to data and tools. 

__Flow:__
- __User Query and Server Discovery:__ The orchestrator receives the query and connects to a running MCP Server via local `stdio` or remote Server-Sent Event.
- __Context Augmentation (MCP Resources):__ The model processes the schema and the query, mapping the intent to a specific tool.
- __Tool Execution:__ The agent invokes the tool, which is exposed through the MCP connection.
- __Output:__ The result is validated and returned to the user.